# Modern Sentiment Classification for Amazon Baby Product Reviews

This notebook demonstrates state-of-the-art sentiment analysis using:
- **Hugging Face Transformers**: Pre-trained BERT models for fine-tuned sentiment analysis
- **Scikit-learn**: Classical ML baselines for comparison
- **Pandas**: Efficient data manipulation

This replaces the outdated TuriCreate approach with modern, production-ready techniques.

In [ ]:
# Install required packages
import subprocess
import sys

packages = [
    "transformers==4.36.0",
    "torch==2.0.0",
    "scikit-learn==1.3.0",
    "pandas==2.0.0",
    "numpy==1.24.0",
    "datasets==2.14.0"
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

In [1]:
import pandas as pd
import numpy as np
import warnings
from typing import Dict, List, Tuple

warnings.filterwarnings('ignore')

## 1. Data Loading and Exploration

In [2]:
# Load data from CSV (replace with your actual data source)
# For this example, we'll create a sample dataset structure

def load_amazon_reviews(file_path: str) -> pd.DataFrame:
    """
    Load Amazon reviews data from CSV.
    
    Args:
        file_path: Path to the CSV file
    
    Returns:
        DataFrame with columns: name, review, rating
    """
    df = pd.read_csv(file_path)
    # Ensure required columns exist
    required_cols = ['name', 'review', 'rating']
    assert all(col in df.columns for col in required_cols), f"Missing required columns: {required_cols}"
    return df


# Example: Load and preview data
# df = load_amazon_reviews('amazon_reviews.csv')
print("Data loading function defined. Replace with actual CSV path.")

In [3]:
def create_sentiment_labels(df: pd.DataFrame, threshold: int = 4) -> pd.DataFrame:
    """
    Create binary sentiment labels based on rating threshold.
    
    Args:
        df: Input DataFrame
        threshold: Rating above/equal to this is positive (1), below is negative (0)
    
    Returns:
        DataFrame with new 'sentiment' column (1=positive, 0=negative)
    """
    df_processed = df.copy()
    
    # Remove neutral ratings (rating == 3)
    df_processed = df_processed[df_processed['rating'] != 3]
    
    # Create binary sentiment: 1 if rating >= threshold, 0 otherwise
    df_processed['sentiment'] = (df_processed['rating'] >= threshold).astype(int)
    
    return df_processed


print("Sentiment label creation function defined.")

## 2. Feature Engineering Using Modern NLP

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

def extract_tfidf_features(texts: List[str], max_features: int = 1000) -> Tuple[np.ndarray, object]:
    """
    Extract TF-IDF features from text using scikit-learn.
    
    Args:
        texts: List of text strings
        max_features: Maximum number of features to extract
    
    Returns:
        Tuple of (feature matrix, vectorizer object)
    """
    vectorizer = TfidfVectorizer(
        max_features=max_features,
        min_df=2,  # Ignore words appearing in < 2 documents
        max_df=0.8,  # Ignore words appearing in > 80% of documents
        ngram_range=(1, 2),  # Use unigrams and bigrams
        stop_words='english'
    )
    
    features = vectorizer.fit_transform(texts)
    return features, vectorizer


print("TF-IDF feature extraction function defined.")

In [5]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch

class ModernSentimentAnalyzer:
    """
    Modern sentiment analyzer using Hugging Face pre-trained models.
    Uses DistilBERT for efficiency.
    """
    
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english"):
        """
        Initialize the sentiment analyzer with a pre-trained model.
        
        Args:
            model_name: HuggingFace model identifier
        """
        self.device = 0 if torch.cuda.is_available() else -1
        self.classifier = pipeline(
            "sentiment-analysis",
            model=model_name,
            device=self.device
        )
    
    def predict_batch(self, texts: List[str], batch_size: int = 32) -> List[Dict]:
        """
        Predict sentiment for a batch of texts.
        
        Args:
            texts: List of text strings
            batch_size: Batch size for processing
        
        Returns:
            List of dictionaries with 'label' and 'score'
        """
        predictions = []
        
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            batch_predictions = self.classifier(batch)
            predictions.extend(batch_predictions)
        
        return predictions
    
    def predict_with_confidence(self, texts: List[str]) -> pd.DataFrame:
        """
        Get predictions with confidence scores.
        
        Args:
            texts: List of text strings
        
        Returns:
            DataFrame with columns: text, prediction, confidence
        """
        predictions = self.predict_batch(texts)
        
        results = []
        for text, pred in zip(texts, predictions):
            results.append({
                'text': text[:100],  # Truncate for display
                'prediction': 1 if pred['label'] == 'POSITIVE' else 0,
                'confidence': pred['score']
            })
        
        return pd.DataFrame(results)


print("ModernSentimentAnalyzer class defined.")

## 3. Classical ML Baseline Models

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

class ClassicalMLBaseline:
    """
    Classical machine learning models for sentiment classification.
    Provides fast baselines for comparison with deep learning models.
    """
    
    def __init__(self, model_type: str = "logistic"):
        """
        Initialize classical ML model.
        
        Args:
            model_type: One of ['logistic', 'random_forest', 'svm']
        """
        self.model_type = model_type
        
        if model_type == "logistic":
            self.model = LogisticRegression(max_iter=1000, random_state=42)
        elif model_type == "random_forest":
            self.model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
        elif model_type == "svm":
            self.model = LinearSVC(random_state=42, max_iter=2000)
        else:
            raise ValueError(f"Unknown model_type: {model_type}")
    
    def train(self, X_train: np.ndarray, y_train: np.ndarray) -> None:
        """
        Train the model.
        
        Args:
            X_train: Training features
            y_train: Training labels
        """
        self.model.fit(X_train, y_train)
    
    def evaluate(self, X_test: np.ndarray, y_test: np.ndarray) -> Dict:
        """
        Evaluate the model on test data.
        
        Args:
            X_test: Test features
            y_test: Test labels
        
        Returns:
            Dictionary with evaluation metrics
        """
        y_pred = self.model.predict(X_test)
        
        metrics = {
            'accuracy': (y_pred == y_test).mean(),
            'classification_report': classification_report(y_test, y_pred),
            'confusion_matrix': confusion_matrix(y_test, y_pred)
        }
        
        # ROC-AUC for models that support decision_function
        if self.model_type in ['logistic', 'svm']:
            try:
                y_scores = self.model.decision_function(X_test)
                metrics['roc_auc'] = roc_auc_score(y_test, y_scores)
            except:
                pass
        
        return metrics
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        """
        Make predictions.
        
        Args:
            X: Features
        
        Returns:
            Predictions
        """
        return self.model.predict(X)


print("ClassicalMLBaseline class defined.")

## 4. Complete Pipeline Example

In [7]:
class SentimentClassificationPipeline:
    """
    End-to-end sentiment classification pipeline.
    Combines data processing, feature engineering, and model training.
    """
    
    def __init__(self, test_size: float = 0.2, random_state: int = 42):
        """
        Initialize the pipeline.
        
        Args:
            test_size: Fraction of data to use for testing
            random_state: Random seed for reproducibility
        """
        self.test_size = test_size
        self.random_state = random_state
        self.vectorizer = None
        self.models = {}
    
    def prepare_data(self, df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        """
        Prepare data: clean reviews and extract features.
        
        Args:
            df: Input DataFrame with 'review' and 'sentiment' columns
        
        Returns:
            Tuple of (features, labels)
        """
        # Clean text: remove NaN, convert to lowercase
        reviews = df['review'].fillna('').str.lower().tolist()
        labels = df['sentiment'].values
        
        # Extract TF-IDF features
        features, self.vectorizer = extract_tfidf_features(reviews)
        
        return features, labels
    
    def train_classical_models(self, X_train: np.ndarray, y_train: np.ndarray) -> Dict:
        """
        Train multiple classical ML models.
        
        Args:
            X_train: Training features
            y_train: Training labels
        
        Returns:
            Dictionary of trained models
        """
        model_types = ['logistic', 'random_forest', 'svm']
        
        for model_type in model_types:
            print(f"Training {model_type} model...")
            model = ClassicalMLBaseline(model_type)
            model.train(X_train, y_train)
            self.models[model_type] = model
        
        return self.models
    
    def evaluate_all_models(self, X_test: np.ndarray, y_test: np.ndarray) -> Dict:
        """
        Evaluate all trained models.
        
        Args:
            X_test: Test features
            y_test: Test labels
        
        Returns:
            Dictionary of evaluation results
        """
        results = {}
        
        for model_name, model in self.models.items():
            print(f"\nEvaluating {model_name}...")
            metrics = model.evaluate(X_test, y_test)
            results[model_name] = metrics
            print(f"Accuracy: {metrics['accuracy']:.4f}")
            if 'roc_auc' in metrics:
                print(f"ROC-AUC: {metrics['roc_auc']:.4f}")
        
        return results


print("SentimentClassificationPipeline class defined.")

## 5. Demonstration with Synthetic Data

In [8]:
# Create synthetic dataset for demonstration
def create_sample_data(n_samples: int = 1000) -> pd.DataFrame:
    """
    Create sample Amazon reviews data for demonstration.
    
    Args:
        n_samples: Number of samples to generate
    
    Returns:
        Sample DataFrame
    """
    positive_reviews = [
        "This product is amazing! Highly recommended.",
        "Excellent quality and fast shipping.",
        "Love it! Great value for money.",
        "Perfect! Exceeded my expectations.",
        "Fantastic product, will buy again."
    ]
    
    negative_reviews = [
        "Terrible quality, waste of money.",
        "Disappointed with this product.",
        "Broke after one day. Not recommended.",
        "Poor customer service and quality.",
        "Worst purchase ever made."
    ]
    
    np.random.seed(42)
    reviews = []
    ratings = []
    
    for _ in range(n_samples // 2):
        reviews.append(np.random.choice(positive_reviews))
        ratings.append(np.random.choice([4, 5]))
        reviews.append(np.random.choice(negative_reviews))
        ratings.append(np.random.choice([1, 2]))
    
    return pd.DataFrame({
        'name': [f'Product_{i}' for i in range(len(reviews))],
        'review': reviews,
        'rating': ratings
    })


# Create and preview sample data
df_sample = create_sample_data(100)
print("Sample data created:")
print(df_sample.head(10))
print(f"\nDataset shape: {df_sample.shape}")
print(f"Rating distribution:\n{df_sample['rating'].value_counts().sort_index()}")

In [9]:
# Process data through pipeline
print("\n" + "="*60)
print("SENTIMENT CLASSIFICATION PIPELINE")
print("="*60)

# Step 1: Create sentiment labels
df_processed = create_sentiment_labels(df_sample)
print(f"\nStep 1: Sentiment Label Creation")
print(f"Data after filtering neutral ratings: {df_processed.shape[0]} samples")
print(f"Sentiment distribution:\n{df_processed['sentiment'].value_counts()}")

# Step 2: Initialize pipeline
pipeline = SentimentClassificationPipeline(test_size=0.2, random_state=42)

# Step 3: Prepare data
print(f"\nStep 2: Feature Engineering (TF-IDF)")
X, y = pipeline.prepare_data(df_processed)
print(f"Feature matrix shape: {X.shape}")
print(f"Number of TF-IDF features: {X.shape[1]}")

# Step 4: Split data
print(f"\nStep 3: Train-Test Split")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Training set positive ratio: {y_train.mean():.2%}")
print(f"Test set positive ratio: {y_test.mean():.2%}")

In [10]:
# Step 5: Train classical models
print(f"\nStep 4: Train Classical ML Models")
pipeline.train_classical_models(X_train, y_train)

# Step 6: Evaluate models
print(f"\nStep 5: Model Evaluation")
results = pipeline.evaluate_all_models(X_test, y_test)

# Display detailed results
print("\n" + "="*60)
print("DETAILED RESULTS")
print("="*60)

for model_name, metrics in results.items():
    print(f"\n{model_name.upper()} CLASSIFIER")
    print("-" * 40)
    print(f"Accuracy: {metrics['accuracy']:.4f}")
    if 'roc_auc' in metrics:
        print(f"ROC-AUC: {metrics['roc_auc']:.4f}")
    print(f"\nConfusion Matrix:")
    print(metrics['confusion_matrix'])
    print(f"\nClassification Report:")
    print(metrics['classification_report'])

## 6. Inference Examples

In [11]:
def predict_sentiment_batch(texts: List[str], model_obj: ClassicalMLBaseline, vectorizer: object) -> pd.DataFrame:
    """
    Predict sentiment for new texts using trained model.
    
    Args:
        texts: List of review texts
        model_obj: Trained model object
        vectorizer: TF-IDF vectorizer
    
    Returns:
        DataFrame with predictions
    """
    # Clean texts
    texts_clean = [text.lower() for text in texts]
    
    # Vectorize using trained vectorizer
    X_new = vectorizer.transform(texts_clean)
    
    # Predict
    predictions = model_obj.predict(X_new)
    
    results = []
    for text, pred in zip(texts, predictions):
        results.append({
            'review': text[:80],
            'prediction': 'POSITIVE' if pred == 1 else 'NEGATIVE',
            'confidence': 'High' if pred in [0, 1] else 'Medium'
        })
    
    return pd.DataFrame(results)


# Test with new reviews
test_reviews = [
    "This product is absolutely fantastic! Love it so much.",
    "Terrible quality. Complete waste of money and time.",
    "Not bad, but could be better for the price.",
    "Amazing! Best purchase I've made in years!"
]

print("\n" + "="*60)
print("INFERENCE ON NEW REVIEWS (Using Logistic Regression)")
print("="*60)

predictions_df = predict_sentiment_batch(
    test_reviews,
    pipeline.models['logistic'],
    pipeline.vectorizer
)

print(predictions_df.to_string(index=False))

## 7. Advanced: Modern Transformer-Based Approach (Optional - Requires GPU)

Uncomment to use state-of-the-art BERT models for even better accuracy.

In [12]:
# # Uncomment to use modern Hugging Face models (requires GPU for speed)
# print("\n" + "="*60)
# print("TRANSFORMER-BASED SENTIMENT ANALYSIS")
# print("="*60)
#
# # Initialize modern sentiment analyzer
# modern_analyzer = ModernSentimentAnalyzer()
#
# print("\nPredictions using DistilBERT:")
# modern_predictions = modern_analyzer.predict_with_confidence(test_reviews)
# print(modern_predictions)
#
# # Compare with classical models
# print("\nModel Comparison:")
# comparison = pd.DataFrame({
#     'review': test_reviews,
#     'logistic': predictions_df['prediction'].values,
#     'transformer': modern_predictions['prediction'].values
# })
# print(comparison.to_string(index=False))

print("Transformer-based approach available but commented (requires GPU).")

## 8. Export Results and Model Saving

In [13]:
import joblib
import json
from pathlib import Path

def save_model_and_vectorizer(model: object, vectorizer: object, output_dir: str) -> None:
    """
    Save trained model and vectorizer for production use.
    
    Args:
        model: Trained model object
        vectorizer: Fitted vectorizer
        output_dir: Directory to save files
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Save model
    model_path = Path(output_dir) / 'sentiment_model.joblib'
    joblib.dump(model, model_path)
    print(f"Model saved to {model_path}")
    
    # Save vectorizer
    vectorizer_path = Path(output_dir) / 'tfidf_vectorizer.joblib'
    joblib.dump(vectorizer, vectorizer_path)
    print(f"Vectorizer saved to {vectorizer_path}")


def load_model_and_vectorizer(model_dir: str) -> Tuple[object, object]:
    """
    Load pre-trained model and vectorizer.
    
    Args:
        model_dir: Directory containing saved files
    
    Returns:
        Tuple of (model, vectorizer)
    """
    model_path = Path(model_dir) / 'sentiment_model.joblib'
    vectorizer_path = Path(model_dir) / 'tfidf_vectorizer.joblib'
    
    model = joblib.load(model_path)
    vectorizer = joblib.load(vectorizer_path)
    
    print(f"Model loaded from {model_path}")
    print(f"Vectorizer loaded from {vectorizer_path}")
    
    return model, vectorizer


# Example: Save the best model
print("\n" + "="*60)
print("MODEL PERSISTENCE")
print("="*60)
print("\nExample: save_model_and_vectorizer(")
print("    model=pipeline.models['logistic'],")
print("    vectorizer=pipeline.vectorizer,")
print("    output_dir='./sentiment_models'")
print(")")

## Summary

This notebook demonstrates:

✅ **Modern NLP Techniques**:
- TF-IDF vectorization with scikit-learn
- Integration with Hugging Face Transformers
- Ready-to-use pre-trained BERT models

✅ **Scalable ML Pipeline**:
- Modular design for easy extension
- Multiple baseline models (Logistic Regression, Random Forest, SVM)
- Comprehensive evaluation metrics

✅ **Production-Ready Code**:
- Type hints for clarity
- Docstrings for documentation
- Model persistence for deployment
- Error handling and validation

✅ **Reproducibility**:
- Fixed random seeds
- Train-test split with stratification
- Detailed results logging

This approach is far superior to TuriCreate as it uses modern libraries actively maintained by the ML community.